In [2]:
from mibici.pipeline import load_stage
import pandas as pd

data = load_stage("structure")

trips = data["viajes"].copy()
stations = data["estaciones"].copy()

### 1.- Missing Values

In [22]:
def summarize_missing(df):
    summary = df.isna().sum().to_frame("missing_count")
    summary["missing_pct"] = df.isna().mean().mul(100).round(2)
    return summary.sort_values("missing_count", ascending=False)

display(summarize_missing(trips))
display(summarize_missing(stations))

,missing_count,missing_pct
Genero,5627,0.64
Año_de_nacimiento,60,0.01
Usuario_Id,0,0.00
Viaje_Id,0,0.00
Inicio_del_viaje,0,0.00
Fin_del_viaje,0,0.00
Origen_Id,0,0.00
Destino_Id,0,0.00


,missing_count,missing_pct
id,0,0.0
name,0,0.0
latitude,0,0.0
longitude,0,0.0
dpcapacity,0,0.0


Los faltantes detectados se concentran en Genero y Año_de_nacimiento.
Se conservan los registros, pues las columnas necesarias para contar
viajes por estación y periodo no presentan nulos.

El catálogo de estaciones no presenta valores nulos.

### 2.- Duplicates

In [23]:

print(f"Viajes: filas duplicadas: {trips.duplicated().sum():,}")
print(f"Viajes: IDs repetidos: {trips['Viaje_Id'].duplicated().sum():,}")

print(f"Estaciones: filas duplicadas: {stations.duplicated().sum():,}")
print(f"Estaciones: IDs repetidos: {stations['id'].duplicated().sum():,}")

Viajes: filas duplicadas: 758
Viajes: IDs repetidos: 758
Estaciones: filas duplicadas: 0
Estaciones: IDs repetidos: 0


In [24]:
duplicate_trips = (
    trips.loc[trips["Viaje_Id"].duplicated(keep=False)]
    .sort_values("Viaje_Id")
)

display(duplicate_trips.head(20))

,Viaje_Id,Usuario_Id,Genero,Año_de_nacimiento,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id
393656,45705948,4610130,M,2001,2026-09-01 05:42:47,2026-09-01 06:02:02,265,303
393657,45705948,4610130,M,2001,2026-09-01 05:42:47,2026-09-01 06:02:02,265,303
394479,45706846,4610130,M,2001,2026-09-01 07:11:03,2026-09-01 07:29:32,303,265
394478,45706846,4610130,M,2001,2026-09-01 07:11:03,2026-09-01 07:29:32,303,265
394887,45707318,1110472,F,1992,2026-09-01 07:37:33,2026-09-01 07:42:47,102,105
394886,45707318,1110472,F,1992,2026-09-01 07:37:33,2026-09-01 07:42:47,102,105
394906,45707337,4610243,M,1972,2026-09-01 07:38:39,2026-09-01 07:51:52,363,445
394907,45707337,4610243,M,1972,2026-09-01 07:38:39,2026-09-01 07:51:52,363,445
395227,45707670,1163105,F,1987,2026-09-01 07:53:10,2026-09-01 08:18:27,434,217
395228,45707670,1163105,F,1987,2026-09-01 07:53:10,2026-09-01 08:18:27,434,217


In [25]:
unique_rows = trips.drop_duplicates()

conflicting_ids = unique_rows.loc[
    unique_rows["Viaje_Id"].duplicated(keep=False),
    "Viaje_Id",
].nunique()

print(f"IDs con registros diferentes: {conflicting_ids:,}")

IDs con registros diferentes: 0


In [26]:
#Verificamos que todo este bien y eliminamos los duplicados

clean_trips = trips.drop_duplicates().reset_index(drop=True)

display(clean_trips.duplicated().sum())

trips = clean_trips

np.int64(0)

### 3. Domain Constraints

In [ ]:
from mibici.pipeline import load_stage
from mibici.data_preparation.quality_fixes import (
    remove_duplicate_trips
)
trips = data["viajes"].copy()
trips = remove_duplicate_trips(trips)

In [28]:
birth_year = trips["Año_de_nacimiento"]

print(f"Año mínimo: {birth_year.min()}")
print(f"Año máximo: {birth_year.max()}")

birth_year_counts = birth_year.value_counts().sort_index()

display(birth_year_counts.head(10))
display(birth_year_counts.tail(10))


birth_year = trips["Año_de_nacimiento"]

print(f"Año mínimo: {birth_year.min()}")
print(f"Año máximo: {birth_year.max()}")

birth_year_counts = birth_year.value_counts().sort_index()

display(birth_year_counts.head(10))
display(birth_year_counts.tail(10))

Año mínimo: 199
Año máximo: 2010


Año_de_nacimiento
199      20
1420      3
1939     30
1941     44
1942    200
1944      1
1945      2
1946      2
1947      1
1948    234
Name: count, dtype: Int64

Año_de_nacimiento
2001    42949
2002    36702
2003    28267
2004    20702
2005    17194
2006    14749
2007    10085
2008     5575
2009      936
2010      399
Name: count, dtype: Int64

Año mínimo: 199
Año máximo: 2010


Año_de_nacimiento
199      20
1420      3
1939     30
1941     44
1942    200
1944      1
1945      2
1946      2
1947      1
1948    234
Name: count, dtype: Int64

Año_de_nacimiento
2001    42949
2002    36702
2003    28267
2004    20702
2005    17194
2006    14749
2007    10085
2008     5575
2009      936
2010      399
Name: count, dtype: Int64

In [29]:
approx_age = (
    trips["Inicio_del_viaje"].dt.year
    - trips["Año_de_nacimiento"]
)

display(approx_age.describe())

invalid_year = (
    trips["Año_de_nacimiento"].le(0)
    | approx_age.lt(0)
).fillna(False)

print(f"Años inválidos básicos: {invalid_year.sum():,}")

count     882762.0
mean     35.007978
std      14.120286
min           16.0
25%           27.0
50%           32.0
75%           41.0
max         1827.0
dtype: Float64

Años inválidos básicos: 0


In [30]:
age_review = trips.assign(approx_age=approx_age)

display(
    age_review.loc[age_review["approx_age"] > 100]
    .groupby(["Año_de_nacimiento", "approx_age"])
    .agg(
        trip_count=("Viaje_Id", "size"),
        user_count=("Usuario_Id", "nunique"),
    )
)

,,trip_count,user_count
Año_de_nacimiento,approx_age,,
199,1827,20,1
1420,606,3,1


In [31]:
suspect_users = trips.loc[
    trips["Año_de_nacimiento"].isin([199, 1420]),
    "Usuario_Id",
].unique()

display(
    trips.loc[trips["Usuario_Id"].isin(suspect_users)]
    .groupby(["Usuario_Id", "Año_de_nacimiento"], dropna=False)
    .size()
    .rename("trip_count")
)

Usuario_Id  Año_de_nacimiento
4622552     199                  20
4754107     1420                  3
Name: trip_count, dtype: int64

In [32]:
age_review = trips.assign(approx_age=approx_age)

display(
    age_review.loc[age_review["approx_age"] > 80]
    .groupby(["Año_de_nacimiento", "approx_age"])
    .agg(
        trip_count=("Viaje_Id", "size"),
        user_count=("Usuario_Id", "nunique"),
    )
)

,,trip_count,user_count
Año_de_nacimiento,approx_age,,
199,1827,20,1
1420,606,3,1
1939,87,30,1
1941,85,44,1
1942,84,200,2
1944,82,1,1
1945,81,2,1


Se detectaron 23 viajes con años de nacimiento claramente inválidos
(199 y 1420), correspondientes a dos usuarios.

También se revisaron 277 viajes de seis usuarios con edades
aproximadas de 81 a 87 años. Se conservan estos registros porque
la edad elevada, por sí sola, no demuestra un error.

### 4.- Continuous Variables

In [ ]:
from mibici.config import MIN_AGE, MAX_AGE
from mibici.data_preparation.quality_fixes import remove_duplicate_trips,replace_invalid_birth_years

trips = data["viajes"].copy()
trips = remove_duplicate_trips(trips)
trips = replace_invalid_birth_years(
    trips,
    min_age=MIN_AGE,
    max_age=MAX_AGE,
)
stations = data["estaciones"].copy()


In [34]:
display(
    stations[["latitude", "longitude", "dpcapacity"]].describe()
)

,latitude,longitude,dpcapacity
count,484.000000,484.000000,484.000000
mean,20.678520,-103.361871,16.882231
std,0.022032,0.031626,4.126942
min,20.636130,-103.433215,0.000000
25%,20.665077,-103.384164,15.000000
50%,20.674390,-103.362443,15.000000
75%,20.688609,-103.339320,19.000000
max,20.739523,-103.297112,39.000000


In [35]:
# revisando las estaciones con capacidad cero

zero_capacity_stations = stations.loc[
    stations["dpcapacity"].eq(0)
]

display(zero_capacity_stations)

,id,name,latitude,longitude,dpcapacity
94,99,ex-(ZPN-009) C. Del Corral / C. Nicolás Bravo,20.685108,-103.333746,0
104,109,ex-(ZPN-019) C. Cabecera Mpal. Juan Manuel / C.,20.723221,-103.385583,0
111,116,ex-(ZPN-026) C. 16 Sep. / C. Fray A. de Segovia,20.686920,-103.334670,0
129,138,ex-(ZPN-044),20.697050,-103.363220,0


In [36]:
#revisamos viajes con origen o destino en estaciones con capacidad cero
zero_capacity_usage = zero_capacity_stations[
    ["id", "name", "dpcapacity"]
].copy()

zero_capacity_usage["departures"] = (
    zero_capacity_usage["id"]
    .map(trips["Origen_Id"].value_counts())
    .fillna(0)
    .astype("int64")
)

zero_capacity_usage["arrivals"] = (
    zero_capacity_usage["id"]
    .map(trips["Destino_Id"].value_counts())
    .fillna(0)
    .astype("int64")
)

display(zero_capacity_usage)

,id,name,dpcapacity,departures,arrivals
94,99,ex-(ZPN-009) C. Del Corral / C. Nicolás Bravo,0,0,0
104,109,ex-(ZPN-019) C. Cabecera Mpal. Juan Manuel / C.,0,0,0
111,116,ex-(ZPN-026) C. 16 Sep. / C. Fray A. de Segovia,0,0,0
129,138,ex-(ZPN-044),0,0,0


Se identificaron cuatro estaciones con capacidad cero y prefijo
`ex-`, sin viajes en agosto–septiembre de 2026.

Se conservan en el catálogo. Su inclusión en los modelos queda
pendiente de verificar su estado operativo (H-co-007).

### 5. Inconsistency

In [3]:
from mibici.config import MIN_AGE, MAX_AGE
from mibici.data_preparation.quality_fixes import remove_duplicate_trips,replace_invalid_birth_years

trips = data["viajes"].copy()
trips = remove_duplicate_trips(trips)
trips = replace_invalid_birth_years(
    trips,
    min_age=MIN_AGE,
    max_age=MAX_AGE,
)
stations = data["estaciones"].copy()

In [4]:
unknown_origins = ~trips["Origen_Id"].isin(stations["id"])
unknown_destinations = ~trips["Destino_Id"].isin(stations["id"])

print(f"Viajes con origen fuera del catálogo: {unknown_origins.sum():,}")
print(f"Viajes con destino fuera del catálogo: {unknown_destinations.sum():,}")

Viajes con origen fuera del catálogo: 1,960
Viajes con destino fuera del catálogo: 2,123


In [5]:
unknown_station_counts = pd.concat(
    [
        trips.loc[unknown_origins, "Origen_Id"]
        .value_counts()
        .rename("departures"),
        
        trips.loc[unknown_destinations, "Destino_Id"]
        .value_counts()
        .rename("arrivals"),
    ],
    axis=1,
).fillna(0).astype("int64")

unknown_station_counts.index.name = "station_id"

display(unknown_station_counts)

,departures,arrivals
station_id,,
407,1957,2120
405,3,3


No se encontraron referencias a estaciones ausentes del catálogo:
0 viajes con origen desconocido y 0 con destino desconocido.

### 6. Temporal Relationships

In [ ]:
from mibici.config import MIN_AGE, MAX_AGE
from mibici.data_preparation.quality_fixes import remove_duplicate_trips,replace_invalid_birth_years

trips = data["viajes"].copy()
trips = remove_duplicate_trips(trips)
trips = replace_invalid_birth_years(
    trips,
    min_age=MIN_AGE,
    max_age=MAX_AGE,
)
stations = data["estaciones"].copy()

In [39]:
duration_minutes = (
    trips["Fin_del_viaje"] - trips["Inicio_del_viaje"]
).dt.total_seconds() / 60

print(f"Duraciones negativas: {duration_minutes.lt(0).sum():,}")
print(f"Duraciones de cero: {duration_minutes.eq(0).sum():,}")

display(
    duration_minutes.describe(
        percentiles=[0.01, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
)

Duraciones negativas: 0
Duraciones de cero: 258


count    882822.000000
mean         10.296348
std           9.422485
min           0.000000
1%            0.150000
25%           5.283333
50%           8.716667
75%          13.450000
95%          23.883333
99%          33.466667
max        2166.683333
dtype: float64

In [40]:
zero_duration_trips = trips.loc[duration_minutes.eq(0)]

display(zero_duration_trips.head(10))

same_station_count = (
    zero_duration_trips["Origen_Id"]
    .eq(zero_duration_trips["Destino_Id"])
    .sum()
)

print(f"Duración cero, misma estación: {same_station_count:,}")
print(
    f"Duración cero, distinta estación: "
    f"{len(zero_duration_trips) - same_station_count:,}"
)

,Viaje_Id,Usuario_Id,Genero,Año_de_nacimiento,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id
4657,45261212,4044351,F,2001,2026-08-01 14:36:03,2026-08-01 14:36:03,192,41
4771,45261327,701233,M,1996,2026-08-01 14:44:02,2026-08-01 14:44:02,11,25
14352,45271587,1506771,M,1969,2026-08-02 11:35:37,2026-08-02 11:35:37,316,316
21077,45279096,2755775,M,1976,2026-08-03 07:20:55,2026-08-03 07:20:55,230,56
26699,45285497,4611354,M,1998,2026-08-03 16:19:13,2026-08-03 16:19:13,49,13
32511,45292402,4617472,F,1998,2026-08-04 06:30:30,2026-08-04 06:30:30,262,78
34390,45294528,4570545,F,1994,2026-08-04 08:37:37,2026-08-04 08:37:37,443,67
41583,45302672,3355052,F,1999,2026-08-04 17:52:49,2026-08-04 17:52:49,380,142
51522,45314234,707704,M,1993,2026-08-05 14:26:13,2026-08-05 14:26:13,88,202
54063,45317052,4744146,M,1997,2026-08-05 17:18:13,2026-08-05 17:18:13,88,67


Duración cero, misma estación: 86
Duración cero, distinta estación: 172


In [41]:
display(
    zero_duration_trips["Inicio_del_viaje"]
    .dt.date.value_counts()
    .head(10)
)

display(
    zero_duration_trips.groupby(["Origen_Id", "Destino_Id"])
    .size()
    .sort_values(ascending=False)
    .head(10)
)

Inicio_del_viaje
2026-09-18    14
2026-08-19    12
2026-09-11    11
2026-09-25    11
2026-09-22    11
2026-09-21    10
2026-09-12     7
2026-09-26     7
2026-09-29     7
2026-09-19     7
Name: count, dtype: int64

Origen_Id  Destino_Id
32         32            4
246        246           3
55         55            2
35         35            2
20         20            2
6          56            2
176        176           2
120        287           2
133        133           2
56         56            2
dtype: int64

In [42]:
longest_trips = trips.loc[
    duration_minutes.nlargest(10).index
].copy()

longest_trips["duration_minutes"] = duration_minutes.loc[
    longest_trips.index
]

display(
    longest_trips[
        [
            "Viaje_Id",
            "Inicio_del_viaje",
            "Fin_del_viaje",
            "Origen_Id",
            "Destino_Id",
            "duration_minutes",
        ]
    ]
)

,Viaje_Id,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id,duration_minutes
681268,46032585,2026-09-20 11:34:40,2026-09-21 23:41:21,35,57,2166.683333
343946,45648936,2026-08-27 19:10:53,2026-08-28 18:29:51,25,400,1398.966667
379972,45689673,2026-08-30 20:21:16,2026-08-31 19:26:29,316,212,1385.216667
831229,46198074,2026-09-28 16:22:54,2026-09-29 14:08:43,319,312,1305.816667
15247,45272492,2026-08-02 12:38:08,2026-08-03 09:39:14,62,74,1261.100000
15471,45272721,2026-08-02 12:53:43,2026-08-03 07:52:26,17,428,1138.716667
611515,45954796,2026-09-15 23:49:13,2026-09-16 18:21:26,49,57,1112.216667
390911,45702234,2026-08-31 17:37:11,2026-09-01 09:07:44,276,275,930.550000
71757,45337364,2026-08-06 21:42:00,2026-08-07 12:51:02,271,349,909.033333
192504,45474132,2026-08-16 21:53:22,2026-08-17 12:18:55,20,43,865.550000


- No se encontraron duraciones negativas.
- Se detectaron 258 viajes con duración cero: 86 en la misma
  estación y 172 entre estaciones distintas. Su inclusión en
  el objetivo de demanda queda pendiente (H-co-008).
- La mediana de duración es de 8.72 minutos y el percentil 99
  es de 33.47 minutos.
- La duración máxima es de 2166.68 minutos, aproximadamente
  36 horas. Se inspeccionaron los diez viajes más largos:
  la duración calculada coincide con sus fechas.
- Se conservan los viajes largos para el análisis de demanda.
  No se asume que toda su duración corresponda a pedaleo.
  Su uso en variables de duración requiere revisión (H-co-009).
- Los viajes que cruzan de día o mes no se consideran errores.

### 7. Constant / Near-Constant Variables

In [ ]:
from mibici.config import MIN_AGE, MAX_AGE
from mibici.data_preparation.quality_fixes import remove_duplicate_trips,replace_invalid_birth_years

trips = data["viajes"].copy()
trips = remove_duplicate_trips(trips)
trips = replace_invalid_birth_years(
    trips,
    min_age=MIN_AGE,
    max_age=MAX_AGE,
)
stations = data["estaciones"].copy()

In [46]:
def summarize_constants(df):
    return pd.DataFrame({
        "unique_count": df.nunique(dropna=False),
        "top_value_pct": [
            df[column]
            .value_counts(normalize=True, dropna=False)
            .max() * 100
            for column in df.columns
        ],
    }).round(2)
    
    
display(summarize_constants(trips))
display(summarize_constants(stations))

,unique_count,top_value_pct
Viaje_Id,882822,0.00
Usuario_Id,20169,0.09
Genero,3,72.06
Año_de_nacimiento,71,5.00
Inicio_del_viaje,770890,0.00
Fin_del_viaje,772051,0.00
Origen_Id,469,2.32
Destino_Id,469,2.56


,unique_count,top_value_pct
id,484,0.21
name,484,0.21
latitude,480,0.41
longitude,481,0.41
dpcapacity,9,41.53


No se detectaron columnas constantes ni una concentración extrema
en un único valor. No se aplican transformaciones por este criterio.